In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# 1. Load and aggregate data (same as Day 1)
df = pd.read_csv(r'C:\Users\ajasl\OneDrive\Documents\Project\The Developers Arena\Month 5\data\supermarket_sales.csv')
df['Date'] = pd.to_datetime(df['Date'])
daily_sales = df.groupby('Date')['Total'].sum().reset_index().set_index('Date').sort_index()

# 2. Scale the data for the Neural Network
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(daily_sales['Total'].values.reshape(-1, 1))

# 3. Create sequences (Use past 7 days to predict the next day)
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length), 0])
        y.append(data[i + seq_length, 0])
    return np.array(X), np.array(y)

seq_length = 7
X, y = create_sequences(scaled_data, seq_length)
X = np.reshape(X, (X.shape[0], X.shape[1], 1)) # Reshape for LSTM [samples, time steps, features]

# 4. Split into train and test sets (80/20)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# 5. Build and train the LSTM model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(seq_length, 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
print("Training the LSTM model...")
history = model.fit(X_train, y_train, batch_size=16, epochs=50, validation_split=0.1, verbose=1)

print("\nDay 2 Model Training Complete. Ready for evaluation and API deployment.")

C:\Users\ajasl\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training the LSTM model...
Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - loss: 0.0977 - val_loss: 0.0338
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0340 - val_loss: 0.0301
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0332 - val_loss: 0.0249
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0329 - val_loss: 0.0251
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0330 - val_loss: 0.0253
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0299 - val_loss: 0.0248
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0284 - val_loss: 0.0255
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0341 - val_loss: 0.0249
Epoch 9/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0303 - val_loss: 0.0252
Epoch 10/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0271 - val_loss: 0.0246
Epoch 11/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0353 - val_loss: 0.0251
Epoch 12/50
16/16 ━━━━━━━━━━━━━━━━━━━

In [3]:
import joblib
import os

# Define the exact folder path
save_dir = r'C:\Users\ajasl\OneDrive\Documents\Project\The Developers Arena\Month 5\models'

# Create the models folder automatically if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Save the scaler
joblib.dump(scaler, os.path.join(save_dir, 'sales_scaler.pkl'))

# Save the trained LSTM model
model.save(os.path.join(save_dir, 'lstm_sales_model.h5'))

print("Model and scaler successfully saved to the models folder!")

Model and scaler successfully saved to the models folder!
